<center>

# **Auto-generate a report and apply a theme**

</center>

### Purpose
Spins up a **simple one-page Power BI report** on top of an existing semantic model and applies a **custom theme** - all from code, no Power BI Desktop required.

Pair this with the *Build Semantic Model with Semantic Link Labs* notebook to demo the full code-first model-and-report flow.

> Requires `semantic-link-labs` (`%pip install semantic-link-labs`).

### Configuration
Point at the semantic model, name the report, and pick which fact / date to visualise. Defaults match the model produced by the partner notebook.

In [ ]:
# Source semantic model
dataset_name = "Sales - Auto Generated"
dataset_workspace = None       # None = current workspace

# Target report
report_name = "Sales Overview - Auto Generated"
report_workspace = None        # None = same as dataset workspace

# What to put on the page (defaults assume the partner notebook's output)
fact_table = "fact_sales"
date_table = "dim_date"
date_column = "Date"
category_table = "dim_product"
category_column = "category"

### Define a custom theme
A small JSON theme - palette + typography + shared visual styles. Power BI applies this to the whole report. Tweak `dataColors` or `name` to make it your own.

In [ ]:
theme = {
    "name": "Workshop Aurora",
    "dataColors": [
        "#3B82F6", "#10B981", "#F59E0B", "#EF4444",
        "#8B5CF6", "#14B8A6", "#F97316", "#64748B",
    ],
    "background": "#FFFFFF",
    "foreground": "#0F172A",
    "tableAccent": "#3B82F6",
    "textClasses": {
        "title":   {"fontSize": 16, "fontFace": "Segoe UI Semibold", "color": "#0F172A"},
        "header":  {"fontSize": 12, "fontFace": "Segoe UI Semibold", "color": "#0F172A"},
        "label":   {"fontSize": 10, "fontFace": "Segoe UI",         "color": "#475569"},
        "callout": {"fontSize": 36, "fontFace": "Segoe UI Light",   "color": "#3B82F6"},
    },
    "visualStyles": {
        "*": {
            "*": {
                "background": [{"color": {"solid": {"color": "#FFFFFF"}}, "transparency": 0}],
                "border":     [{"color": {"solid": {"color": "#E2E8F0"}}, "radius": 6}],
            }
        }
    },
}

### Build the report definition
Constructs a minimal one-page report:
- A **slicer** on the date table
- Two **KPI cards** for the auto-generated sales measures
- A **clustered column chart** of sales by category
- A **line chart** of sales over time

The shape used here is the public **PBIR** (Power BI Project / report.json) format. `semantic-link-labs` ships helpers for creating reports from a JSON definition, which we use below.

In [ ]:
import json
import sempy_labs as labs
from sempy_labs import report as labs_report


def visual_container(name, x, y, width, height, visual_type, projections, title=None):
    """Tiny helper that builds a visual container in the legacy report.json shape."""
    config = {
        "name": name,
        "layouts": [{"id": 0, "position": {"x": x, "y": y, "z": 0, "width": width, "height": height}}],
        "singleVisual": {
            "visualType": visual_type,
            "projections": projections,
            "prototypeQuery": {"Version": 2, "From": [], "Select": []},
            "drillFilterOtherVisuals": True,
        },
    }
    if title:
        config["singleVisual"]["vcObjects"] = {
            "title": [{"properties": {"text": {"expr": {"Literal": {"Value": f"'{title}'"}}}}}]
        }
    return {"config": json.dumps(config)}


page = {
    "name": "ReportSection1",
    "displayName": "Sales Overview",
    "width": 1280,
    "height": 720,
    "displayOption": 1,
    "visualContainers": [
        visual_container("slicer-date",  20,  20, 280,  90, "slicer",
                         {"Field": [{"queryRef": f"{date_table}.{date_column}"}]},
                         title="Date"),
        visual_container("card-rows",   320,  20, 280, 140, "card",
                         {"Values": [{"queryRef": f"{fact_table}.# {fact_table}"}]},
                         title=f"# {fact_table}"),
        visual_container("card-amount", 620,  20, 280, 140, "card",
                         {"Values": [{"queryRef": f"{fact_table}.Total Amount"}]},
                         title="Total Amount"),
        visual_container("col-by-cat",   20, 180, 600, 260, "clusteredColumnChart",
                         {
                             "Category": [{"queryRef": f"{category_table}.{category_column}"}],
                             "Y":        [{"queryRef": f"{fact_table}.Total Amount"}],
                         },
                         title="Total Amount by Category"),
        visual_container("line-overtime", 640, 180, 620, 260, "lineChart",
                         {
                             "Category": [{"queryRef": f"{date_table}.{date_column}"}],
                             "Y":        [{"queryRef": f"{fact_table}.Total Amount"}],
                         },
                         title="Total Amount over Time"),
    ],
}

report_json = {
    "config": json.dumps({"version": "5.43", "themeCollection": {"baseTheme": {"name": "CY24SU10"}}}),
    "layoutOptimization": 0,
    "sections": [page],
    "resourcePackages": [],
}

print(f"Report definition built with {len(page['visualContainers'])} visuals on 1 page.")

### Create the report bound to the semantic model
Calls `create_report_from_reportjson` to publish the report against the existing dataset.

In [ ]:
labs_report.create_report_from_reportjson(
    report=report_name,
    dataset=dataset_name,
    report_json=report_json,
    workspace=report_workspace or dataset_workspace,
)

print(f"Report '{report_name}' created.")

### Apply the custom theme
Pushes the inline theme JSON onto the freshly created report. Re-running this cell with a tweaked `theme` is the fastest way to iterate on look & feel.

In [ ]:
labs_report.set_report_theme(
    report=report_name,
    workspace=report_workspace or dataset_workspace,
    theme_json=theme,
)

print(f"Theme '{theme['name']}' applied to '{report_name}'.")

### Done
Open the report in the workspace - one page, a handful of visuals, branded theme - all built from code in seconds.

_Note: SLL's report APIs evolve quickly. If a method signature has changed, fall back to the workspace REST API (`POST /v1.0/myorg/groups/{id}/reports`) or `update_report_theme` and re-run._